In [29]:
import pandas as pd
import polars as pl
import fsspec
import plotly.express as px
import ast
import json
import numpy as np

In [30]:
#import polars as pl

#df = pl.read_parquet('hf://datasets/fddemarco/pushshift-reddit/**/*.parquet')


[Link al dataset](https://huggingface.co/datasets/finiteautomata/news-argentina)

In [31]:


fs, _, paths = fsspec.get_fs_token_paths("hf://datasets/fddemarco/pushshift-reddit/**/*.parquet")
files = fs.ls(paths[0])

print(files)

[{'name': 'datasets/fddemarco/pushshift-reddit/data/RS_2012-01_00.parquet', 'size': 112243140, 'type': 'file', 'blob_id': 'ab40a869c7772c2e0dafdba11c3b4fba59739fc0', 'lfs': BlobLfsInfo(size=112243140, sha256='e88737be217579ae52349073d5166dc601a1e882e012078fcfd6b2a25625487c', pointer_size=134), 'last_commit': None, 'security': None}]


In [32]:

fs, _, paths = fsspec.get_fs_token_paths("hf://datasets/fddemarco/pushshift-reddit/**/*.parquet")
files_info = fs.ls(paths[0])

# me quedo solo con los nombres (paths completos dentro de hf://)
files = [f"hf://{f['name']}" for f in files_info]

print(files)

['hf://datasets/fddemarco/pushshift-reddit/data/RS_2012-01_00.parquet']


In [33]:
fs, _, paths = fsspec.get_fs_token_paths("hf://datasets/fddemarco/pushshift-reddit/**/*.parquet")
files_info = fs.ls(paths[0])

# Extraigo solo los nombres en formato completo hf://...
files = [f"hf://{f['name']}" for f in files_info if f['name'].endswith(".parquet")]

# Ordeno por nombre (para que respete el orden 00000, 00001, 00002...)
files = sorted(files)

print("Archivos encontrados:")
for f in files:
    print(f)

Archivos encontrados:
hf://datasets/fddemarco/pushshift-reddit/data/RS_2012-01_00.parquet


In [34]:
lf = pl.scan_parquet(
    files,
    cast_options=pl.ScanCastOptions(extra_struct_fields="ignore")
)

pdf = lf.collect()

In [35]:
df = pdf.to_pandas()

print(df.info())

df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   author        800000 non-null  object
 1   created_utc   800000 non-null  int64 
 2   id            800000 non-null  object
 3   num_comments  800000 non-null  int64 
 4   score         800000 non-null  int64 
 5   selftext      800000 non-null  object
 6   subreddit     800000 non-null  object
 7   subreddit_id  800000 non-null  object
 8   title         800000 non-null  object
dtypes: int64(3), object(6)
memory usage: 54.9+ MB
None


,author,created_utc,id,num_comments,score,selftext,subreddit,subreddit_id,title
0,doopercooper,1325462399,nz3g9,24,82,,videos,t5_2qh1e,guy chases after runaway car and barely saves ...
1,,1325462398,nz3g8,0,0,,gaming,t5_2qh03,the legend of zelda rap by smosh
2,thiswebpage,1325462398,nz3g7,16,77,,hiphopheads,t5_2rh4c,how i feel about most rap battles
3,belifted,1325462395,nz3g6,0,4,,funny,t5_2qh33,eliot chang spread the word
4,,1325462392,nz3g5,0,1,,leagueoflegends,t5_2rfxx,mordekaiser bored screwin around on ps


In [36]:
df.author.value_counts()

author
                      225442
bonafera                2065
davidreiss666           1943
redditteam              1562
iamtotalcrap            1516
                       ...  
pprt5                      1
yunus89115                 1
insuranceclaiminsp         1
raslavernon                1
rbcp                       1
Name: count, Length: 244990, dtype: int64

In [37]:
print(df.subreddit.value_counts().to_string())

subreddit
funny                    51971
pics                     34552
AdviceAnimals            31861
AskReddit                30360
fffffffuuuuuuuuuuuu      26913
trees                    20159
politics                 18334
videos                   17826
gaming                   17821
WTF                      15871
atheism                  12318
Music                    11509
technology               10275
aww                       9911
worldnews                 9753
circlejerk                8453
todayilearned             8235
reportthespammers         5802
skyrim                    5524
leagueoflegends           5448
SteamGameSwap             5407
tf2trade                  5157
firstworldproblems        5144
starcraft                 4731
general                   4596
movies                    4542
Minecraft                 4349
askscience                4237
mylittlepony              4135
bestof                    4125
gonewild                  3701
science                   361

In [38]:
## Transformar Data y crear columnas complementarias
# transformar created_at a yyyy-mm-dd
df["Fecha"] = pd.to_datetime(df["created_utc"], unit="s").dt.date



In [39]:
df.head()

,author,created_utc,id,num_comments,score,selftext,subreddit,subreddit_id,title,Fecha
0,doopercooper,1325462399,nz3g9,24,82,,videos,t5_2qh1e,guy chases after runaway car and barely saves ...,2012-01-01
1,,1325462398,nz3g8,0,0,,gaming,t5_2qh03,the legend of zelda rap by smosh,2012-01-01
2,thiswebpage,1325462398,nz3g7,16,77,,hiphopheads,t5_2rh4c,how i feel about most rap battles,2012-01-01
3,belifted,1325462395,nz3g6,0,4,,funny,t5_2qh33,eliot chang spread the word,2012-01-01
4,,1325462392,nz3g5,0,1,,leagueoflegends,t5_2rfxx,mordekaiser bored screwin around on ps,2012-01-01


In [40]:
print(df.Fecha.min(), 
df.Fecha.max())

2012-01-01 2012-01-13
